
# 05 Final Load Prep — KPI Tables and Dashboard Feed

## Objective
Prepare analytics-ready exports from the cleaned dataset for Tableau/reporting:
- master record-level dataset
- KPI summary table
- country, market, and yearly aggregate tables


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)


In [2]:

PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'startups_cleaned.csv'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_PATH, parse_dates=['founded_at', 'first_funding_at', 'last_funding_at'])

if 'is_closed' not in df.columns:
    df['is_closed'] = (df['status'].str.lower() == 'closed').astype(int)
if 'reached_series_a' not in df.columns:
    df['reached_series_a'] = (df.get('round_a', 0).fillna(0) > 0).astype(int)
if 'avg_funding_per_round' not in df.columns:
    df['avg_funding_per_round'] = np.where(
        (df['funding_rounds'] > 0) & (df['funding_total_usd'] > 0),
        df['funding_total_usd'] / df['funding_rounds'],
        np.nan
    )
if 'days_to_first_funding' not in df.columns:
    df['days_to_first_funding'] = (pd.to_datetime(df['first_funding_at'], errors='coerce') - pd.to_datetime(df['founded_at'], errors='coerce')).dt.days

# Final semantic enrichments for BI
if 'primary_category' not in df.columns:
    df['primary_category'] = df['category_list'].astype('string').str.strip('|').str.split('|').str[0]

df['founding_decade'] = (df['founded_year'] // 10 * 10).astype('Int64').astype('string') + 's'

print(f'Loaded for final prep: {len(df):,} rows')


for dcol in ['founded_at', 'first_funding_at', 'last_funding_at']:
    if dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors='coerce')

for dcol in ['founded_at', 'first_funding_at', 'last_funding_at']:
    if dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors='coerce')


Loaded for final prep: 49,437 rows


## 1) KPI Summary

In [3]:

total = len(df)
closed = int((df['status'] == 'closed').sum())
operating = int((df['status'] == 'operating').sum())
acquired = int((df['status'] == 'acquired').sum())
ipo_count = int((df['status'] == 'ipo').sum())

kpi_summary = pd.DataFrame({
    'KPI': [
        'Total Startups Analysed',
        'Overall Failure Rate (%)',
        'Operating Rate (%)',
        'Acquisition Rate (%)',
        'IPO Rate (%)',
        'Median Funding Total (USD)',
        'Median Funding Rounds',
        'Series A Reach Rate (%)'
    ],
    'Value': [
        total,
        round(closed / total * 100, 2),
        round(operating / total * 100, 2),
        round(acquired / total * 100, 2),
        round(ipo_count / total * 100, 2),
        round(df['funding_total_usd'].median(skipna=True), 2),
        round(df['funding_rounds'].median(skipna=True), 2),
        round(df['reached_series_a'].mean(skipna=True) * 100, 2)
    ]
})

kpi_summary


,KPI,Value
0,Total Startups Analysed,49437.00
1,Overall Failure Rate (%),5.27
2,Operating Rate (%),84.61
3,Acquisition Rate (%),7.47
4,IPO Rate (%),0.00
5,Median Funding Total (USD),2000000.00
6,Median Funding Rounds,1.00
7,Series A Reach Rate (%),18.21


## 2) Aggregated Tables for Dashboards

In [4]:

country_level = (
    df.groupby('country_code')
    .agg(
        Total_Startups=('is_closed', 'size'),
        Closed=('is_closed', 'sum'),
        Avg_Funding_USD=('funding_total_usd', 'mean'),
        Median_Funding_USD=('funding_total_usd', 'median'),
        Avg_Rounds=('funding_rounds', 'mean')
    )
    .reset_index()
)
country_level['Failure_Rate_%'] = (country_level['Closed'] / country_level['Total_Startups'] * 100).round(2)
country_level = country_level.sort_values(['Total_Startups', 'Failure_Rate_%'], ascending=[False, False])

sector_level = (
    df.groupby('market')
    .agg(
        Total_Startups=('is_closed', 'size'),
        Closed=('is_closed', 'sum'),
        Avg_Funding_USD=('funding_total_usd', 'mean'),
        Avg_Rounds=('funding_rounds', 'mean')
    )
    .reset_index()
)
sector_level['Failure_Rate_%'] = (sector_level['Closed'] / sector_level['Total_Startups'] * 100).round(2)
sector_level = sector_level.sort_values(['Total_Startups', 'Failure_Rate_%'], ascending=[False, False])

yearly_trend = (
    df[df['founded_year'].between(1990, 2015)]
    .groupby('founded_year')
    .agg(
        Total_Startups=('is_closed', 'size'),
        Closed=('is_closed', 'sum'),
        Avg_Funding_USD=('funding_total_usd', 'mean')
    )
    .reset_index()
)
yearly_trend['Failure_Rate_%'] = (yearly_trend['Closed'] / yearly_trend['Total_Startups'] * 100).round(2)

print('country_level shape:', country_level.shape)
print('sector_level shape :', sector_level.shape)
print('yearly_trend shape :', yearly_trend.shape)


country_level shape: (115, 7)
sector_level shape : (753, 6)
yearly_trend shape : (25, 5)


## 3) Final Record-Level Master Table

In [5]:

master_cols = [
    'name', 'status', 'status_group', 'country_code', 'state_code', 'region', 'city',
    'market', 'primary_category', 'founded_year', 'founding_decade',
    'founded_at', 'first_funding_at', 'last_funding_at',
    'funding_total_usd', 'funding_rounds', 'avg_funding_per_round',
    'round_a', 'round_b', 'round_c', 'round_d',
    'seed', 'venture', 'angel', 'grant',
    'days_to_first_funding', 'funding_duration_days',
    'reached_series_a', 'is_closed'
]

existing_cols = [c for c in master_cols if c in df.columns]
master = df[existing_cols].copy()

print('master shape:', master.shape)
master.head()


master shape: (49437, 29)


,name,status,status_group,country_code,state_code,region,city,market,primary_category,founded_year,founding_decade,founded_at,first_funding_at,last_funding_at,funding_total_usd,funding_rounds,avg_funding_per_round,round_a,round_b,round_c,round_d,seed,venture,angel,grant,days_to_first_funding,funding_duration_days,reached_series_a,is_closed
0,#waywire,acquired,terminal,USA,NY,New York City,New York,News,Entertainment,2012.0,2010s,2012-06-01,2012-06-30,2012-06-30,1750000.0,1.0,1750000.0,0.0,0.0,0.0,0.0,1750000.0,0.0,0.0,0.0,29.0,0.0,0,0.0
1,&TV Communications,operating,active,USA,CA,Los Angeles,Los Angeles,Games,Games,NaN,<NA>,NaT,2010-06-04,2010-09-23,4000000.0,2.0,2000000.0,0.0,0.0,0.0,0.0,0.0,4000000.0,0.0,0.0,NaN,111.0,0,0.0
2,'Rock' Your Paper,operating,active,EST,NaN,Tallinn,Tallinn,Publishing,Publishing,2012.0,2010s,2012-10-26,2012-08-09,2012-08-09,40000.0,1.0,40000.0,0.0,0.0,0.0,0.0,40000.0,0.0,0.0,0.0,-78.0,0.0,0,0.0
3,(In)Touch Network,operating,active,GBR,NaN,London,London,Electronics,Electronics,2011.0,2010s,2011-04-01,2011-04-01,2011-04-01,1500000.0,1.0,1500000.0,0.0,0.0,0.0,0.0,1500000.0,0.0,0.0,0.0,0.0,0.0,0,0.0
4,-R- Ranch and Mine,operating,active,USA,TX,Dallas,Fort Worth,Tourism,Tourism,2014.0,2010s,2014-01-01,2014-08-17,2014-09-26,60000.0,2.0,30000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,228.0,40.0,0,0.0


## 4) Export Final Load Files

In [6]:

files = {
    'tableau_master.csv': master,
    'kpi_summary.csv': kpi_summary,
    'country_level_summary.csv': country_level,
    'sector_level_summary.csv': sector_level,
    'yearly_trend_summary.csv': yearly_trend,
}

for fname, data in files.items():
    out = OUT_DIR / fname
    data.to_csv(out, index=False)
    print(f'Exported: {out}')


Exported: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/tableau_master.csv
Exported: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/kpi_summary.csv
Exported: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/country_level_summary.csv
Exported: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/sector_level_summary.csv
Exported: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/yearly_trend_summary.csv



## Final Load Conclusion
- All KPI and aggregate tables are now generated from the cleaned source.
- The outputs are consistent with EDA/statistical notebooks and ready for dashboard consumption.
